# 🎤 YingMusic-SVC — Google Colab

เปลี่ยนเสียงร้องในเพลงให้เป็นเสียงของคนอื่น ใช้แค่ตัวอย่างเสียง 10–25 วินาที · *Make any song sung in another voice from a 10–25 s sample*

| ขั้น | ทำอะไร |
|---|---|
| 0 | `Runtime → Change runtime type → T4 GPU` |
| 1 | กด ▶ **ติดตั้ง** (ครั้งแรก ~3–5 นาที) |
| 2 | กด ▶ **เปิดหน้าเว็บ** แล้วใช้งานในหน้าต่างที่โผล่ขึ้นมา: อัปโหลดไฟล์ / อัดเสียง / วางลิงก์ YouTube ได้หมด |
| 2′ | หรือถ้าไม่อยากใช้หน้าเว็บ ใช้ช่อง **โหมดฟอร์ม** ด้านล่างแทน |

> หน้าเว็บจะเปิดอยู่ใน notebook นี้เลย ไม่ต้องใช้ลิงก์สาธารณะ (ติ๊ก `SHARE_LINK` เฉพาะตอนอยากเปิดในแท็บใหม่หรือมือถือ)

In [ ]:
#@title 1️⃣ ติดตั้ง / Install
#@markdown กด ▶ ครั้งเดียวต่อ session · ช่องด้านล่างปกติไม่ต้องแก้
REPO_URL = "https://github.com/DDME36/YingMusic-SVC" #@param {type:"string"}
BRANCH = "" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

import os, sys, shlex, subprocess, importlib

def sh(cmd, cwd=None):
    """รันคำสั่ง แสดง output สด และหยุดถ้าล้มเหลว"""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise SystemExit(f"❌ ล้มเหลว (exit {p.returncode}): {cmd}")

q = shlex.quote
REPO_DIR = "/content/YingMusic-SVC"
READY = "/content/.ymsvc_ready_v5"

if subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone -q --depth 1 {'-b ' + q(BRANCH) if BRANCH else ''} {q(REPO_URL)} {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR) if REPO_DIR not in sys.path else None
sh('git log -1 --format="code version: %h %s"')

if not os.path.exists(READY):
    print("📦 กำลังติดตั้ง dependencies ... (ข้อความเตือนของ pip ข้ามได้)")
    # Colab มี torch/torchaudio/ffmpeg อยู่แล้ว ติดตั้งเฉพาะที่ขาดและไม่แตะ torch เพื่อไม่ให้เวอร์ชันชนกัน
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y sox > /dev/null")
    sh("pip install -q -U -r requirements-app.txt")
    import svc_pipeline
    print("⬇️ กำลังโหลดโมเดล ...")
    svc_pipeline.ensure_checkpoints(HF_TOKEN.strip())
    open(READY, "w").close()

# โหลดโค้ดล่าสุดเสมอ (เผื่อ git pull มีของใหม่)
import svc_pipeline, gradio_app
importlib.reload(svc_pipeline); importlib.reload(gradio_app)
sh("python -c \"import torch, torchaudio; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| CUDA', torch.cuda.is_available())\"")
print("✅ พร้อมใช้งาน — ไปที่ขั้นที่ 2")

In [ ]:
#@title 2️⃣ เปิดหน้าเว็บ / Open the web UI
LANGUAGE = "ไทย" #@param ["ไทย", "English"]
#@markdown ติ๊กถ้าอยากได้ลิงก์ `gradio.live` ไว้เปิดในแท็บใหม่หรือบนมือถือ (ปกติไม่ต้อง)
SHARE_LINK = False #@param {type:"boolean"}

import gradio as gr
gr.close_all()
demo = gradio_app.build_ui(LANGUAGE).queue()
demo.launch(share=SHARE_LINK, inline=True, height=1300, quiet=True)

In [ ]:
#@title 2′ โหมดฟอร์ม (ไม่เปิดหน้าเว็บ) / Form mode
#@markdown **เพลงต้นฉบับ** — วางลิงก์ YouTube หรือเว้นว่างไว้แล้วจะมีปุ่มให้อัปโหลดไฟล์
SONG_URL = "" #@param {type:"string"}
#@markdown ตัดช่วงเพลง (วินาที): ความยาว 0 = ทั้งเพลง · ลอง 30 วิก่อนจะเร็วกว่ามาก
CLIP_START = 0 #@param {type:"number"}
CLIP_LENGTH = 0 #@param {type:"number"}
#@markdown ติ๊กถ้าเพลงมีดนตรีประกอบ (จะแยกเสียงร้องออกก่อน)
HAS_ACCOMPANIMENT = True #@param {type:"boolean"}
KEEP_BACKING_VOCAL = False #@param {type:"boolean"}
#@markdown ---
#@markdown **เสียงเป้าหมาย** — `upload` = อัปโหลดไฟล์, `youtube` = ใช้ลิงก์ด้านล่าง, หรือเลือกเสียงตัวอย่าง
TIMBRE = "upload" #@param ["upload", "youtube", "tp", "syz"]
TIMBRE_URL = "" #@param {type:"string"}
TIMBRE_START = 0 #@param {type:"number"}
#@markdown ติ๊กถ้าเสียงเป้าหมายมีดนตรีปน (เช่นเอามาจากเพลง)
TIMBRE_HAS_MUSIC = False #@param {type:"boolean"}
#@markdown ---
DIFFUSION_STEPS = 100 #@param {type:"slider", min:10, max:200, step:10}
#@markdown คีย์: ว่าง = อัตโนมัติ · +12 = สูงขึ้น 1 ออกเทฟ (ชาย→หญิง) · -12 = ต่ำลง (หญิง→ชาย)
PITCH_SHIFT = "" #@param {type:"string"}

import os, shutil
from google.colab import files
from IPython.display import Audio, display
P = svc_pipeline
work = P.new_workdir("form")

def upload_one(prompt):
    print(prompt)
    up = files.upload()
    if not up:
        raise SystemExit("❌ ไม่ได้อัปโหลดไฟล์")
    name = next(iter(up))
    dst = os.path.join(work, "upload_" + P.safe_name(os.path.splitext(name)[0]) + os.path.splitext(name)[1])
    shutil.move(name, dst)
    return dst

try:
    song_src = None if SONG_URL.strip() else upload_one("🎵 อัปโหลดเพลงต้นฉบับ (mp3/wav/flac/m4a/mp4):")
    song_wav, title = P.fetch_audio(song_src, SONG_URL, work, "song", start=CLIP_START, duration=CLIP_LENGTH)

    if TIMBRE in P.SAMPLE_TIMBRES:
        timbre_wav = P.SAMPLE_TIMBRES[TIMBRE]
    else:
        t_src = upload_one("🗣️ อัปโหลดเสียงเป้าหมาย (10–25 วินาที):") if TIMBRE == "upload" else None
        t_wav, _ = P.fetch_audio(t_src, TIMBRE_URL if TIMBRE == "youtube" else "", work, "timbre_raw",
                                 start=TIMBRE_START, duration=P.TIMBRE_MAX_SEC)
        if TIMBRE_HAS_MUSIC:
            t_wav = P.separate(t_wav)["vocals"]
        timbre_wav = P.to_wav(t_wav, os.path.join(work, "timbre.wav"), mono=True)

    res = P.cover(song_wav, timbre_wav, work, has_accompaniment=HAS_ACCOMPANIMENT, keep_backing=KEEP_BACKING_VOCAL,
                  steps=DIFFUSION_STEPS, pitch_shift=int(PITCH_SHIFT) if PITCH_SHIFT.strip() else None)
except P.PipelineError as e:
    msg = str(e)
    if msg == "YOUTUBE_BOT_CHECK":
        msg = "YouTube บล็อกการดาวน์โหลด — ใช้หน้าเว็บ (ขั้น 2) แล้วอัปโหลด cookies.txt หรืออัปโหลดไฟล์เพลงเอง"
    raise SystemExit(f"❌ {msg}")

name = P.safe_name(title, "song")
for key, label in (("cover", "🎧 เพลงคัฟเวอร์"), ("vocal", "🗣️ เสียงร้องที่แปลงแล้ว")):
    if res[key]:
        final = os.path.join(work, f"{name}_{key}.wav")
        os.replace(res[key], final)
        print(f"\n{label}: {final}")
        display(Audio(filename=P.to_mp3(final)))
        files.download(final)